# Yelp Human-vs-AI 멀티모달 분류 (RoBERTa + Hand-craft 양방향 Cross-Attention)

`yelp_dataset/data_yelp.parquet` (20,000행, balanced `human`/`ai`, 원본 7컬럼 + 23 hand-craft feature)로 **label 이진 분류**.

**모델 (시퀀스↔시퀀스 Bidirectional cross-attention, ViLBERT co-TRM 방식):**
- text(RoBERTa 마지막 hidden layer) → `Dense(d)` → `T [B,256,d]`
- HC → `FeatureTokenizer`(FT-Transformer 방식) → `F [B,23,d]` — feature 1개 = 토큰 1개
- 방향 1: `MHA(Q=F, K=V=T, 패딩마스크)` → 평균 → `co_text` / 방향 2: `MHA(Q=T, K=V=F)` → masked mean → `co_hc`
- `Concat[co_text, co_hc]` → MLP → softmax(2) — co-attention 출력만 융합

> 사전 요약 벡터 없이 padding mask + FeatureTokenizer만으로 양방향을 구성.
> mask 역할 2가지: 방향1 attention의 패딩 K/V 차단, 방향2 출력 풀링의 패딩 쿼리 제외.

**환경:** transformers 5.x + M2 Pro (CUDA 없음) → RoBERTa는 PyTorch frozen 임베딩 1회 추출·캐싱, 분류기는 TF/Keras.

**MAX_LEN=256** — 토큰 분포(p95=241, p99=329) 기준. 128은 31.7% 잘림 + human이 ai보다 길어 라벨 편향.

### 1. 임포트 & 설정
> 사전 설치(한 번만): `uv pip install --python Yelp/.venv/bin/python tensorflow scikit-learn`

In [1]:
# ⚠️ 중요: TensorFlow 를 *가장 먼저* import 해야 한다 (pandas/numpy 보다도 먼저).
#   1) pandas 3.x는 import 시 pyarrow 를 로드하는데, pyarrow 와 TF 가 각자 내장한 abseil 의
#      심볼이 충돌해 TF 첫 fit() 의 absl::Mutex 가 Arrow 쪽 구현에 바인딩 → 영구 데드락 (CPU 0%).
#   2) sklearn(scipy)/torch 의 OpenMP 선로드 문제도 동일하게 TF 선임포트로 회피.
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks

import os, time, itertools, warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
import pandas as pd

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

BASE      = ".."
DATA      = f"{BASE}/data/data_yelp.parquet"
EMB_DIR   = f"{BASE}/data/roberta_emb"
MODEL_DIR = f"{BASE}/models"
os.makedirs(EMB_DIR, exist_ok=True); os.makedirs(MODEL_DIR, exist_ok=True)

GRID_CSV = f"{MODEL_DIR}/coattn_grid_results.csv"
TEST_CSV = f"{MODEL_DIR}/coattn_best_test.csv"
WEIGHTS  = f"{MODEL_DIR}/coattn_best.weights.h5"

ROBERTA_NAME = "roberta-base"
# 토큰 길이 분포(roberta-base, balanced 4K 샘플): mean 113, p90 203, p95 241, p99 329, max 397.
# MAX_LEN=128로 자르면 31.7%가 잘리고, human(p95=290)이 ai(p95=197)보다 길어
# truncation이 라벨 신호를 선택적으로 깎아냄 → 256으로 두면 잘림 4%, 라벨 편향 거의 제거.
MAX_LEN = 256
HIDDEN  = 768

# 임베딩 캐시는 MAX_LEN을 파일명에 박아 128/256이 공존 가능.
EMB_PATH  = f"{EMB_DIR}/last_hidden_{MAX_LEN}.npy"
MASK_PATH = f"{EMB_DIR}/mask_{MAX_LEN}.npy"

# === 단일 조합 고정: lr=1e-4, dropout=0.3, d_model=256, batch=32, heads=4 ===
# (grid search 미수행 — 전체 탐색으로 되돌리려면:
#   LR_GRID=[3e-5,1e-4,3e-4], DROPOUT_GRID=[0.0,0.1,0.3], DMODEL_GRID=[128,256,512], BATCH_GRID=[32,64,128])
LR_GRID      = [1e-4]
DROPOUT_GRID = [0.3]
DMODEL_GRID  = [256]
BATCH_GRID   = [32]
NUM_HEADS  = 4
MAX_EPOCHS = 15
PATIENCE   = 3

# SMOKE=True → 그리드 1조합·데이터 일부·짧은 epoch 로 파이프라인만 빠르게 검증.
SMOKE = False
print("SMOKE:", SMOKE, "| MAX_LEN:", MAX_LEN)

TF: 2.21.0 | GPU: []
SMOKE: False | MAX_LEN: 256


### 2. 데이터 로드 & 라벨 매핑

In [2]:
ORIG = ["pk", "review_id", "text", "label", "source", "review_stars", "business_id"]
df = pd.read_parquet(DATA)
FEATURE_COLS = [c for c in df.columns if c not in ORIG]
assert len(FEATURE_COLS) == 23, f"feature 수 이상: {len(FEATURE_COLS)}"

LABEL_MAP = {"human": 0, "ai": 1}
df["y"] = df["label"].map(LABEL_MAP).astype("int64")
assert df["y"].isin([0, 1]).all(), "라벨 매핑 실패"

print("shape:", df.shape, "| n_features:", len(FEATURE_COLS))
print("label dist:", df["y"].value_counts().to_dict())

shape: (20000, 31) | n_features: 23
label dist: {0: 10000, 1: 10000}


### 3. Stratified split 70 / 15 / 15

In [3]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(df))
y_all = df["y"].values
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=y_all)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=y_all[temp_idx])

if SMOKE:
    # 라벨 균형 유지하며 서브샘플 (rng.choice는 stratify를 풀어버리므로 train_test_split 사용).
    def _strat(ix, n):
        _, s = train_test_split(ix, test_size=n, random_state=SEED, stratify=y_all[ix])
        return s
    train_idx = _strat(train_idx, 1400)
    val_idx   = _strat(val_idx,   300)
    test_idx  = _strat(test_idx,  300)

print("train/val/test:", len(train_idx), len(val_idx), len(test_idx))
for name, ix in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"  {name} label dist:", pd.Series(y_all[ix]).value_counts().to_dict())

train/val/test: 14000 3000 3000
  train label dist: {0: 7000, 1: 7000}
  val label dist: {0: 1500, 1: 1500}
  test label dist: {1: 1500, 0: 1500}


### 4. Hand-craft feature 표준화 (train에만 fit)

In [4]:
from sklearn.preprocessing import StandardScaler

HC = df[FEATURE_COLS].astype("float32").values
HC = np.nan_to_num(HC, nan=0.0, posinf=0.0, neginf=0.0)
scaler = StandardScaler().fit(HC[train_idx])
HC_scaled = scaler.transform(HC).astype("float32")
print("HC_scaled:", HC_scaled.shape)

HC_scaled: (20000, 23)


### 5. RoBERTa 임베딩 캐시 (MAX_LEN 변경 시 1회만 실행)
- `EMB_PATH`/`MASK_PATH`가 존재하면 skip. 없으면 `df["text"]` 행 순서 그대로 추출해 디스크에 저장.
- 디바이스: MPS 가능하면 MPS, 아니면 CPU. roberta-base, frozen, `last_hidden_state`만 float16으로 dump.
- **행 순서 일치 검증**도 같은 셀에서 수행.

In [5]:
if not (os.path.exists(EMB_PATH) and os.path.exists(MASK_PATH)):
    import torch
    from transformers import AutoTokenizer, AutoModel

    tok = AutoTokenizer.from_pretrained(ROBERTA_NAME)
    mdl = AutoModel.from_pretrained(ROBERTA_NAME).eval()
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    mdl = mdl.to(device)
    print(f"extracting RoBERTa embeddings → {EMB_PATH} (device={device}, N={len(df)}, L={MAX_LEN})")

    N = len(df); BATCH = 32
    emb_out  = np.empty((N, MAX_LEN, HIDDEN), dtype=np.float16)
    mask_out = np.empty((N, MAX_LEN),         dtype=np.int8)
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, N, BATCH):
            batch = df["text"].iloc[s:s+BATCH].tolist()
            enc = tok(batch, max_length=MAX_LEN, padding="max_length",
                      truncation=True, return_tensors="pt").to(device)
            out = mdl(**enc).last_hidden_state.cpu().numpy().astype(np.float16)
            emb_out[s:s+BATCH]  = out
            mask_out[s:s+BATCH] = enc["attention_mask"].cpu().numpy().astype(np.int8)
            if s % (BATCH * 50) == 0:
                print(f"  {s}/{N}  ({time.time()-t0:.0f}s)")
    np.save(EMB_PATH,  emb_out)
    np.save(MASK_PATH, mask_out)
    print(f"cached in {time.time()-t0:.0f}s → {EMB_PATH}")

    # --- 행 순서 정합성 검증: 임베딩 캐시의 mask가 df.iloc[i]의 토크나이즈 결과와 일치하는지 ---
    _verify_tok = tok
    _MASK = np.load(MASK_PATH, mmap_mode="r")
    for i in [0, len(df) // 2, len(df) - 1]:
        re = _verify_tok(df["text"].iloc[i], max_length=MAX_LEN, padding="max_length",
                         truncation=True, return_tensors="pt")
        assert (re["attention_mask"].numpy().astype(np.int8) == _MASK[i]).all(), \
            f"행 순서 불일치 @ i={i}"
    print("row alignment OK (df.iloc[i] ↔ MASK[i])")
else:
    print(f"cache exists, skip: {EMB_PATH}")

cache exists, skip: /Users/user/project/Yelp/yelp_dataset/roberta_emb/last_hidden_256.npy


### 6. split별 배열 적재

In [6]:
EMB  = np.load(EMB_PATH,  mmap_mode="r")
MASK = np.load(MASK_PATH, mmap_mode="r")
Y    = df["y"].values.astype("int64")

def gather(ix):
    ix = np.asarray(ix)
    emb = np.asarray(EMB[ix],  dtype="float16")   # [n,L,768]
    msk = np.asarray(MASK[ix], dtype="float32")   # [n,L]
    return ({"text_emb": emb, "mask": msk, "hc": HC_scaled[ix]}, Y[ix])

Xtr, ytr = gather(train_idx)
Xva, yva = gather(val_idx)
Xte, yte = gather(test_idx)
print("train emb:", Xtr["text_emb"].shape, Xtr["text_emb"].dtype)

train emb: (14000, 256, 768) float16


### 7. 모델 정의 — 시퀀스↔시퀀스 양방향 Cross-Attention (ViLBERT co-TRM 방식)
- **text**: RoBERTa **마지막 hidden layer** → `Dense(d)` → `T [B,256,d]`
- **HC**: `FeatureTokenizer`(FT-Transformer 방식)로 feature 23개를 각각 토큰화 → `F [B,23,d]` — MLP 요약 없음
- **방향 1 (HC→text)**: `MHA(Q=F, K=V=T, 패딩 K/V 마스크)` → `[B,23,d]` → 평균 → `co_text`
- **방향 2 (text→HC)**: `MHA(Q=T, K=V=F)` → `[B,256,d]` → `MaskedMeanPool`(패딩 쿼리 제외) → `co_hc`
- 최종 `Concat[co_text, co_hc] [B,2d] → MLP → softmax(2)` — **co-attention 출력 2개만으로 분류**

> padding mask는 두 곳에 쓰임: 방향1의 attention_mask(패딩 K/V 차단), 방향2의 풀링(패딩 쿼리 출력 제외).
> 사전 요약 벡터(hc_vec/text_vec) 없이 시퀀스가 그대로 쿼리가 되는 구조 — ViLBERT의 co-attentional
> transformer와 동일한 형태이며, tabular feature 토큰화는 FT-Transformer(NeurIPS 2021) 방식.

In [7]:
import keras  # keras.ops 사용 (KerasTensor를 Functional API에서 cast/슬라이싱)


class MaskedMeanPool(layers.Layer):
    """[B,L,d], mask [B,L] → [B,d]. 패딩 토큰 제외 평균."""
    def call(self, x, mask):
        m = tf.cast(mask, x.dtype)[..., None]
        return tf.reduce_sum(x * m, axis=1) / (tf.reduce_sum(m, axis=1) + 1e-9)


class FeatureTokenizer(layers.Layer):
    """[B,n] → [B,n,d]. HC feature 하나하나를 토큰으로 임베딩 (FT-Transformer 방식).
    out[b,i,:] = x[b,i] * W[i,:] + b[i,:]  — feature별 고유 임베딩이라 정체성 보존."""
    def __init__(self, n_feat, d_model, **kw):
        super().__init__(**kw)
        self.n_feat, self.d_model = n_feat, d_model
    def build(self, _):
        self.W = self.add_weight(name="W", shape=(self.n_feat, self.d_model), initializer="glorot_uniform")
        self.B = self.add_weight(name="B", shape=(self.n_feat, self.d_model), initializer="zeros")
    def call(self, x):                      # [B, n]
        return x[..., None] * self.W + self.B   # [B, n, d]


def build_model(d_model, dropout):
    """시퀀스↔시퀀스 양방향 cross-attention (ViLBERT co-TRM 방식).
    사전 요약 벡터 없이 FeatureTokenizer + padding mask만으로 양방향을 구성하고,
    co_text·co_hc 두 출력만 융합해 분류한다."""
    text_in = layers.Input((MAX_LEN, HIDDEN), dtype="float16", name="text_emb")
    mask_in = layers.Input((MAX_LEN,),        dtype="float32", name="mask")
    hc_in   = layers.Input((len(FEATURE_COLS),), dtype="float32", name="hc")

    # text: RoBERTa "마지막 hidden layer"(last_hidden_state 캐시) → Dense(d) 토큰별 투영
    T = layers.Dense(d_model)(keras.ops.cast(text_in, "float32"))      # [B, L, d]

    # HC: feature 23개를 각각 토큰으로 임베딩 (MLP 요약 없음 — FT-Transformer 방식)
    F = FeatureTokenizer(len(FEATURE_COLS), d_model)(hc_in)            # [B, 23, d]

    # --- 양방향 Cross-Attention (시퀀스가 그대로 쿼리) ---
    # 방향 1: HC → text. HC 토큰 23개가 각자 텍스트를 attend (패딩 K/V 차단)
    key_mask = keras.ops.cast(mask_in, "bool")[:, None, :]             # [B, 1, L] → 23개 쿼리에 브로드캐스트
    A1 = layers.MultiHeadAttention(NUM_HEADS, d_model // NUM_HEADS, name="hc2text")(
        query=F, value=T, key=T, attention_mask=key_mask)              # [B, 23, d]
    co_text = layers.GlobalAveragePooling1D()(A1)                      # [B, d]  (23개 전부 실제 토큰 → 단순 평균)

    # 방향 2: text → HC. 텍스트 토큰 256개가 각자 HC 토큰을 attend
    A2 = layers.MultiHeadAttention(NUM_HEADS, d_model // NUM_HEADS, name="text2hc")(
        query=T, value=F, key=F)                                       # [B, L, d]
    co_hc = MaskedMeanPool()(A2, mask_in)                              # [B, d]  (패딩 위치 쿼리 출력은 풀링에서 제외)

    # 융합 → 분류: 양방향 co-attention 출력만 사용
    x = layers.Concatenate()([co_text, co_hc])                         # [B, 2d]
    x = layers.Dense(d_model, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(2, activation="softmax")(x)
    return Model([text_in, mask_in, hc_in], out, name="hc_bixattn")

### 8. Grid Search (81 조합) — best = val accuracy
각 조합 학습 후 val 예측으로 accuracy/precision/recall/f1 기록, **매 조합마다 CSV 저장**.

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if SMOKE:
    grids = ([1e-4], [0.1], [128], [64]); max_epochs = 2
else:
    grids = (LR_GRID, DROPOUT_GRID, DMODEL_GRID, BATCH_GRID); max_epochs = MAX_EPOCHS
combos = list(itertools.product(*grids))
print("combos:", len(combos))

results, best_acc = [], -1.0
for k, (lr, dr, dm, bs) in enumerate(combos, 1):
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_model(dm, dr)
    model.compile(optimizers.Adam(lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    es = callbacks.EarlyStopping("val_loss", patience=PATIENCE, restore_best_weights=True)

    t0 = time.time()
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                     epochs=max_epochs, batch_size=bs, callbacks=[es], verbose=0)
    sec = time.time() - t0

    pred = model.predict(Xva, batch_size=256, verbose=0).argmax(1)
    row = dict(lr=lr, dropout=dr, d_model=dm, batch=bs,
               epochs_ran=len(hist.history["loss"]), train_sec=round(sec, 1),
               val_accuracy=accuracy_score(yva, pred),
               val_precision=precision_score(yva, pred, zero_division=0),
               val_recall=recall_score(yva, pred, zero_division=0),
               val_f1=f1_score(yva, pred, zero_division=0))
    results.append(row)
    pd.DataFrame(results).to_csv(GRID_CSV, index=False)  # 중단 보존
    print(f"[{k:>2}/{len(combos)}] lr={lr:<6} dr={dr} d={dm:<3} bs={bs:<3} "
          f"val_acc={row['val_accuracy']:.4f} f1={row['val_f1']:.4f} ({sec:.0f}s)")
    if row["val_accuracy"] > best_acc:
        best_acc = row["val_accuracy"]; model.save_weights(WEIGHTS)

res_df = pd.DataFrame(results)
res_df["is_best"] = (res_df["val_accuracy"] == res_df["val_accuracy"].max()).astype(int)
bi = res_df["val_accuracy"].idxmax()
res_df.to_csv(GRID_CSV, index=False)
print("\nBEST (val):", res_df.loc[bi].to_dict())

combos: 1


[ 1/1] lr=0.0001 dr=0.3 d=256 bs=32  val_acc=0.9897 f1=0.9896 (517s)

BEST (val): {'lr': 0.0001, 'dropout': 0.3, 'd_model': 256.0, 'batch': 32.0, 'epochs_ran': 13.0, 'train_sec': 517.2, 'val_accuracy': 0.9896666666666667, 'val_precision': 0.9932840832773674, 'val_recall': 0.986, 'val_f1': 0.9896286383405821, 'is_best': 1.0}


### 9. best 조합 → test set 평가

In [9]:
b = res_df.loc[bi]
best_model = build_model(int(b.d_model), float(b.dropout))
best_model.load_weights(WEIGHTS)

pred = best_model.predict(Xte, batch_size=256, verbose=0).argmax(1)
test_metrics = dict(
    lr=float(b.lr), dropout=float(b.dropout), d_model=int(b.d_model), batch=int(b.batch),
    test_accuracy=accuracy_score(yte, pred),
    test_precision=precision_score(yte, pred, zero_division=0),
    test_recall=recall_score(yte, pred, zero_division=0),
    test_f1=f1_score(yte, pred, zero_division=0))
pd.DataFrame([test_metrics]).to_csv(TEST_CSV, index=False)
print("TEST:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in test_metrics.items()})
print("\nsaved:", GRID_CSV, "|", TEST_CSV, "|", WEIGHTS)

TEST: {'lr': 0.0001, 'dropout': 0.3, 'd_model': 256, 'batch': 32, 'test_accuracy': 0.987, 'test_precision': 0.9893, 'test_recall': 0.9847, 'test_f1': 0.987}

saved: /Users/user/project/Yelp/models/coattn_grid_results.csv | /Users/user/project/Yelp/models/coattn_best_test.csv | /Users/user/project/Yelp/models/coattn_best.weights.h5


### 10. Feature Attention Weight 분석
방향2(text→HC)의 attention score로 **텍스트 토큰들이 23개 HC feature 중 어디에 주목하는지** 집계.
- 학습된 레이어를 재사용하는 probe 모델로 `return_attention_scores=True` 출력 추출
- 균등분배 기준선 = 1/23 ≈ 0.043 — 그보다 큰 feature가 모델이 실제로 활용하는 지표
- human/ai 라벨별 평균을 비교해 attention이 입력에 따라 동적으로 변하는지 확인

In [ ]:
import matplotlib.pyplot as plt

# 학습된 레이어 핸들 (probe용 함수형 Model 재구성은 Keras3에서 그래프 연결 오류 → eager 배치 루프 사용)
dense_T = next(l for l in best_model.layers
               if isinstance(l, layers.Dense) and l.kernel.shape[0] == HIDDEN)
ft   = next(l for l in best_model.layers if isinstance(l, FeatureTokenizer))
mha2 = best_model.get_layer("text2hc")   # 방향2: Text→HC

# === feature별 attention weight 집계 (방향2: 각 토큰이 23개 feature에 분배한 가중치) ===
N, BS = len(yte), 256
feat_attn = np.empty((N, len(FEATURE_COLS)), dtype="float32")   # [N, 23] (행 합=1)
for s in range(0, N, BS):
    e  = tf.cast(tf.convert_to_tensor(Xte["text_emb"][s:s+BS]), tf.float32)
    mk = tf.convert_to_tensor(Xte["mask"][s:s+BS])
    hc = tf.convert_to_tensor(Xte["hc"][s:s+BS])
    T = dense_T(e)
    F = ft(hc)
    _, s2 = mha2(query=T, value=F, key=F, return_attention_scores=True, training=False)
    w = tf.reduce_mean(s2, axis=1)                       # head 평균 → [b, L, 23]
    m = mk[..., None]
    fa = tf.reduce_sum(w * m, axis=1) / tf.reduce_sum(m, axis=1)   # 실토큰 평균 (패딩 쿼리 제외)
    feat_attn[s:s+BS] = fa.numpy()

imp = pd.Series(feat_attn.mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False)
print("=== 전체 평균 attention weight (text→HC, uniform=%.4f) ===" % (1 / len(FEATURE_COLS)))
print(imp.round(4).to_string())

# === 라벨별 비교: 입력에 따라 attention이 동적으로 변하는지 ===
attn_df = pd.DataFrame(feat_attn, columns=FEATURE_COLS)
attn_df["y"] = yte
by_label = attn_df.groupby("y").mean().T
by_label.columns = ["human", "ai"]
by_label["diff(ai-human)"] = by_label["ai"] - by_label["human"]
by_label = by_label.loc[imp.index]   # 중요도순 정렬
display(by_label.round(4))

fig, ax = plt.subplots(figsize=(9, 8))
ypos = np.arange(len(by_label))
ax.barh(ypos - 0.2, by_label["human"], height=0.38, color="#4C72B0", label="human")
ax.barh(ypos + 0.2, by_label["ai"],    height=0.38, color="#DD8452", label="ai")
ax.set_yticks(ypos); ax.set_yticklabels(by_label.index)
ax.invert_yaxis()
ax.axvline(1 / len(FEATURE_COLS), color="gray", ls="--", lw=0.8, label="uniform (1/23)")
ax.set_xlabel("mean attention weight (text→HC)")
ax.set_title("Which HC features do text tokens attend to?")
ax.legend()
fig.tight_layout()
fig.savefig(f"{MODEL_DIR}/feature_attention.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# === Attention weight 표: 학술지(3선 표, booktabs) 스타일 — Times New Roman serif ===
# 앞 셀의 imp(전체 평균), by_label(human/ai/diff)을 그대로 사용.

tbl = by_label.copy()
tbl.insert(0, "Overall", imp)
tbl = tbl.rename(columns={"diff(ai-human)": "Δ (AI−Human)", "human": "Human", "ai": "AI"})
tbl.index.name = "Feature"

CORNELL_STYLE = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
}

with plt.rc_context(CORNELL_STYLE):
    n = len(tbl)
    fig, ax = plt.subplots(figsize=(8.2, 0.34 * n + 1.3))
    ax.axis("off")

    cols = ["Rank", "Feature", "Overall", "Human", "AI", "Δ (AI−Human)"]
    xs   = [0.03, 0.12, 0.48, 0.63, 0.76, 0.93]   # 컬럼 x 위치
    top, bottom = 0.92, 0.04
    row_h = (top - bottom) / (n + 1)

    # 헤더
    for x, c in zip(xs, cols):
        ax.text(x, top - row_h * 0.5, c, ha="left" if c in ("Rank", "Feature") else "right",
                va="center", fontsize=11, fontweight="bold", transform=ax.transAxes)

    # 데이터 행 (상위 3개 feature는 굵게)
    for i, (feat, r) in enumerate(tbl.iterrows()):
        y = top - row_h * (i + 1.5)
        fw = "bold" if i < 3 else "normal"
        vals = [str(i + 1), feat, f"{r['Overall']:.4f}", f"{r['Human']:.4f}",
                f"{r['AI']:.4f}", f"{r['Δ (AI−Human)']:+.4f}"]
        for x, v, c in zip(xs, vals, cols):
            ax.text(x, y, v, ha="left" if c in ("Rank", "Feature") else "right",
                    va="center", fontsize=10.5, fontweight=fw, transform=ax.transAxes)

    # 3선 (booktabs): toprule(굵게) / midrule(가늘게) / bottomrule(굵게)
    for y, lw in [(top, 1.6), (top - row_h, 0.7), (bottom, 1.6)]:
        ax.plot([0.01, 0.99], [y, y], color="black", lw=lw,
                transform=ax.transAxes, clip_on=False)

    # 캡션(표 위)
    ax.text(0.01, top + 0.045, "Table 1. Mean cross-attention weights over hand-craft features (Text→HC direction).",
            ha="left", va="bottom", fontsize=11.5, transform=ax.transAxes)

    fig.savefig(f"{MODEL_DIR}/feature_attention_table.png", dpi=150, bbox_inches="tight")
    plt.show()

print("saved:", f"{MODEL_DIR}/feature_attention_table.png")

### 11. Text 기준 attention 시각화 (방향1: HC→Text)

방향1의 attention score `s1 [B, heads, 23, L]`을 집계하면 **"HC feature들이 리뷰의 어떤 토큰을 주목하는가"** 를
단어 단위로 볼 수 있다. heads(4)×HC쿼리(23) 평균 → 토큰별 가중치 → 리뷰 원문에 주황색 하이라이트(진할수록 높은 attention).

- 샘플: 정분류된 human 2건 + ai 2건 (가독성 위해 120토큰 미만)
- 토크나이즈는 임베딩 캐시와 동일 설정으로 재수행하고 mask 일치를 assert로 검증

In [ ]:
import html as _html
from IPython.display import HTML, display
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(ROBERTA_NAME)
mha1 = best_model.get_layer("hc2text")   # 방향1: HC→Text (dense_T/ft는 섹션 10에서 정의)

# --- 샘플 선정: 정분류된 human 2건 + ai 2건 (120토큰 미만) ---
ix = np.asarray(test_idx)
correct = pred == yte                     # pred는 섹션 9에서 계산됨
tok_len = Xte["mask"].sum(1)
cands_h = np.where(correct & (yte == 0) & (tok_len < 120))[0]
cands_a = np.where(correct & (yte == 1) & (tok_len < 120))[0]
sel = np.r_[cands_h[:2], cands_a[:2]]

LABEL = {0: "human", 1: "ai"}
for li in sel:
    gi = ix[li]                           # 원본 df 인덱스
    text = df["text"].iloc[gi]
    enc = tok(text, max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="np")
    assert (enc["attention_mask"][0].astype("float32") == Xte["mask"][li]).all(), f"mask 불일치 @ {gi}"

    # eager 호출로 방향1 attention score 추출
    e  = tf.cast(tf.convert_to_tensor(Xte["text_emb"][li:li+1]), tf.float32)
    mk = tf.convert_to_tensor(Xte["mask"][li:li+1])
    hc = tf.convert_to_tensor(Xte["hc"][li:li+1])
    T = dense_T(e); F = ft(hc)
    key_mask = tf.cast(mk, tf.bool)[:, None, :]
    _, s1 = mha1(query=F, value=T, key=T, attention_mask=key_mask,
                 return_attention_scores=True, training=False)   # [1, heads, 23, L]
    tw = tf.reduce_mean(s1, axis=(1, 2)).numpy()[0]              # heads·feature 평균 → [L]

    n = int(Xte["mask"][li].sum())
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0][:n])
    w = tw[:n]
    a = (w - w.min()) / (w.max() - w.min() + 1e-12)              # min-max 정규화 → 투명도

    spans = []
    for t_, al in zip(toks, a):
        if t_ in ("<s>", "</s>"):
            continue
        word = t_.replace("Ġ", " ")
        spans.append(f"<span style='background: rgba(221,132,82,{al:.2f})'>{_html.escape(word)}</span>")
    display(HTML(
        f"<h4 style='font-family:Georgia,serif'>label={LABEL[yte[li]]} | pred={LABEL[pred[li]]} | tokens={n}</h4>"
        f"<p style='font-family:Georgia,serif; line-height:1.9; max-width:780px'>{''.join(spans)}</p><hr>"
    ))